# 03 - Exploratory Data Analysis (EDA)

This notebook explores the cleaned Helpdesk Tickets dataset and answers key business questions about ticket volume, departments, categories, priority, technician performance, SLA compliance, satisfaction, and time patterns.

Dataset used: `../data/tickets_helpdesk_clean.csv`

## Setup

We import pandas and matplotlib, load the cleaned dataset, and prepare a few display settings for readable tables and charts.

Some source columns contain French accents. To keep the code simple and avoid encoding problems, the notebook automatically converts column names to simple ASCII names after loading.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import unicodedata

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

DATA_PATH = Path("../data/tickets_helpdesk_clean.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Cleaned dataset not found: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)

# Convert column names such as Departement/Categorie/Priorite to safe ASCII names.
def clean_column_name(name):
    normalized = unicodedata.normalize("NFKD", name)
    ascii_name = normalized.encode("ascii", "ignore").decode("ascii")
    return ascii_name.replace(" ", "_")

df.columns = [clean_column_name(col) for col in df.columns]

# Convert date columns again after loading from CSV, because CSV files do not preserve datetime types.
df["Date_Ouverture"] = pd.to_datetime(df["Date_Ouverture"], errors="coerce")
df["Date_Fermeture"] = pd.to_datetime(df["Date_Fermeture"], errors="coerce")

# Make sure SLA_Binaire is numeric for SLA calculations.
df["SLA_Binaire"] = pd.to_numeric(df["SLA_Binaire"], errors="coerce")
df["SLA_Result"] = df["SLA_Binaire"].map({1: "Respected", 0: "Exceeded"})

print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]:,} columns")
df.head()

## Section 1 - General KPIs

This section gives a high-level overview of the helpdesk activity: total ticket volume, organizational coverage, average resolution time, SLA performance, and customer satisfaction.

In [ ]:
total_tickets = len(df)
number_departments = df["Departement"].nunique()
number_technicians = df["Technicien"].nunique()
avg_resolution_time = df["Temps_Resolution_h"].mean()
sla_compliance_rate = df["SLA_Binaire"].mean() * 100
avg_satisfaction = df["Satisfaction"].mean()

kpi_summary = pd.DataFrame({
    "KPI": [
        "Total number of tickets",
        "Number of departments",
        "Number of technicians",
        "Average resolution time (hours)",
        "SLA compliance rate (%)",
        "Average satisfaction score"
    ],
    "Value": [
        total_tickets,
        number_departments,
        number_technicians,
        round(avg_resolution_time, 2),
        round(sla_compliance_rate, 2),
        round(avg_satisfaction, 2)
    ]
})

kpi_summary

## Section 2 - Department Analysis

This section identifies which departments generate the most helpdesk tickets. High-volume departments may need more support, training, or better self-service documentation.

In [ ]:
department_counts = df["Departement"].value_counts().sort_values(ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(department_counts.index, department_counts.values, color="#2E86AB")
plt.title("Tickets by Department")
plt.xlabel("Number of Tickets")
plt.ylabel("Department")
plt.tight_layout()
plt.show()

**Business interpretation:** Departments at the top of the ranking generate the highest ticket volume. These departments should be reviewed first when looking for recurring issues or opportunities to reduce support demand.

In [ ]:
department_ranking = df["Departement"].value_counts().reset_index()
department_ranking.columns = ["Departement", "ticket_count"]
department_ranking

## Section 3 - Category Analysis

This section analyzes the most frequent ticket categories and subcategories. It helps identify the most common types of helpdesk problems.

In [ ]:
category_counts = df["Categorie"].value_counts().head(10).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(category_counts.index, category_counts.values, color="#3A7D44")
plt.title("Top Ticket Categories")
plt.xlabel("Number of Tickets")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

**Business interpretation:** The most frequent categories represent the main sources of support demand. These areas are good candidates for process improvement, user training, or automation.

In [ ]:
subcategory_counts = df["Sous_Categorie"].value_counts().head(10).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(subcategory_counts.index, subcategory_counts.values, color="#8E5EA2")
plt.title("Top Ticket Subcategories")
plt.xlabel("Number of Tickets")
plt.ylabel("Subcategory")
plt.tight_layout()
plt.show()

**Business interpretation:** Subcategories show the specific problems behind each broad category. The most frequent subcategories can reveal recurring incidents that should be investigated in detail.

## Section 4 - Priority Analysis

This section studies the distribution of ticket priorities and calculates the percentage of each priority level.

In [ ]:
priority_counts = df["Priorite"].value_counts()
priority_percent = (priority_counts / priority_counts.sum() * 100).round(2)

priority_summary = pd.DataFrame({
    "ticket_count": priority_counts,
    "percentage": priority_percent
})

priority_summary

In [ ]:
priority_plot = priority_counts.sort_values(ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(priority_plot.index, priority_plot.values, color="#F18F01")
plt.title("Distribution of Ticket Priorities")
plt.xlabel("Number of Tickets")
plt.ylabel("Priority")
plt.tight_layout()
plt.show()

**Business interpretation:** The priority distribution shows whether the helpdesk is mostly handling low-risk requests or a large number of urgent incidents. A high share of high-priority tickets may indicate operational pressure or recurring critical failures.

In [ ]:
plt.figure(figsize=(7, 7))
plt.pie(priority_counts.values, labels=priority_counts.index, autopct="%1.1f%%", startangle=90)
plt.title("Percentage of Tickets by Priority")
plt.tight_layout()
plt.show()

**Business interpretation:** The percentage view makes it easier to compare the relative weight of each priority level in the total workload.

## Section 5 - Technician Performance

This section compares technicians using three indicators: average resolution time, SLA compliance rate, and number of tickets handled.

In [ ]:
technician_performance = df.groupby("Technicien").agg(
    tickets_handled=("Ticket_ID", "count"),
    avg_resolution_time=("Temps_Resolution_h", "mean"),
    sla_compliance_rate=("SLA_Binaire", lambda x: x.mean() * 100)
).round(2).sort_values("tickets_handled", ascending=False)

technician_performance

In [ ]:
avg_resolution_by_tech = technician_performance.sort_values("avg_resolution_time", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(avg_resolution_by_tech.index, avg_resolution_by_tech["avg_resolution_time"], color="#4D908E")
plt.title("Average Resolution Time by Technician")
plt.xlabel("Average Resolution Time (hours)")
plt.ylabel("Technician")
plt.tight_layout()
plt.show()

**Business interpretation:** Technicians with lower average resolution times close tickets faster. Differences should be interpreted together with ticket volume and priority mix, because some technicians may handle more complex cases.

In [ ]:
sla_by_tech = technician_performance.sort_values("sla_compliance_rate", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(sla_by_tech.index, sla_by_tech["sla_compliance_rate"], color="#277DA1")
plt.title("SLA Compliance Rate by Technician")
plt.xlabel("SLA Compliance Rate (%)")
plt.ylabel("Technician")
plt.xlim(0, 100)
plt.tight_layout()
plt.show()

**Business interpretation:** Higher SLA compliance means a technician resolves a greater share of tickets within the expected time. Low compliance may require workload review, coaching, or investigation of ticket complexity.

In [ ]:
tickets_by_tech = technician_performance.sort_values("tickets_handled", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(tickets_by_tech.index, tickets_by_tech["tickets_handled"], color="#577590")
plt.title("Tickets Handled by Technician")
plt.xlabel("Number of Tickets")
plt.ylabel("Technician")
plt.tight_layout()
plt.show()

**Business interpretation:** Ticket volume by technician helps evaluate workload distribution. Large differences may show uneven assignment of tickets across the support team.

## Section 6 - SLA Analysis

This section looks at SLA risk by category and priority. It helps identify where SLA breaches happen most often.

In [ ]:
sla_exceeded_by_category = (
    df[df["SLA_Binaire"] == 0]
    .groupby("Categorie")
    .size()
    .sort_values(ascending=False)
)

sla_exceeded_by_category

In [ ]:
sla_exceeded_plot = sla_exceeded_by_category.head(10).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(sla_exceeded_plot.index, sla_exceeded_plot.values, color="#C44536")
plt.title("Categories with the Most SLA Breaches")
plt.xlabel("Number of SLA Breaches")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

**Business interpretation:** Categories with the most SLA breaches are the highest-risk service areas. These categories may need faster escalation rules, better documentation, or additional technical resources.

In [ ]:
sla_by_priority = pd.crosstab(df["Priorite"], df["SLA_Result"], normalize="index") * 100
sla_by_priority = sla_by_priority.round(2)
sla_by_priority

In [ ]:
sla_by_priority.plot(kind="bar", figsize=(9, 5), color=["#C44536", "#2E8B57"])
plt.title("SLA Result by Priority")
plt.xlabel("Priority")
plt.ylabel("Percentage of Tickets")
plt.xticks(rotation=0)
plt.legend(title="SLA Result")
plt.tight_layout()
plt.show()

**Business interpretation:** Comparing SLA by priority shows whether urgent tickets are being resolved within targets. If high-priority tickets have many breaches, the support process may need stronger escalation controls.

## Section 7 - Satisfaction Analysis

This section studies the relationship between resolution time and satisfaction, then compares satisfaction by communication channel.

In [ ]:
resolution_satisfaction_corr = df["Temps_Resolution_h"].corr(df["Satisfaction"])
print(f"Correlation between resolution time and satisfaction: {resolution_satisfaction_corr:.3f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["Temps_Resolution_h"], df["Satisfaction"], alpha=0.35, color="#6A4C93")
plt.title("Resolution Time vs Satisfaction")
plt.xlabel("Resolution Time (hours)")
plt.ylabel("Satisfaction Score")
plt.tight_layout()
plt.show()

**Business interpretation:** The correlation and scatter plot show whether longer resolution times are associated with lower satisfaction. A negative correlation would suggest that faster ticket resolution improves user experience.

In [ ]:
satisfaction_by_channel = df.groupby("Canal")["Satisfaction"].mean().sort_values(ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(satisfaction_by_channel.index, satisfaction_by_channel.values, color="#43AA8B")
plt.title("Average Satisfaction by Communication Channel")
plt.xlabel("Average Satisfaction Score")
plt.ylabel("Channel")
plt.xlim(0, 5)
plt.tight_layout()
plt.show()

**Business interpretation:** Satisfaction by channel shows which communication methods provide the best user experience. Lower-scoring channels may need process improvements or clearer response expectations.

In [ ]:
satisfaction_by_channel.to_frame("average_satisfaction")

## Section 8 - Time Analysis

This section explores when tickets are opened by month, weekday, and hour. These patterns can support staffing and scheduling decisions.

In [ ]:
tickets_by_month = df.groupby("Mois").size().sort_index()

plt.figure(figsize=(9, 5))
plt.plot(tickets_by_month.index, tickets_by_month.values, marker="o", color="#1D3557")
plt.title("Tickets by Month")
plt.xlabel("Month")
plt.ylabel("Number of Tickets")
plt.xticks(tickets_by_month.index)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

**Business interpretation:** Monthly ticket trends show seasonal workload changes. Months with higher volume may require extra staffing or preventive actions before demand increases.

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
tickets_by_weekday = df["Jour_Semaine"].value_counts().reindex(weekday_order).fillna(0)

plt.figure(figsize=(10, 5))
plt.bar(tickets_by_weekday.index, tickets_by_weekday.values, color="#457B9D")
plt.title("Tickets by Weekday")
plt.xlabel("Weekday")
plt.ylabel("Number of Tickets")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

**Business interpretation:** Weekday volume helps identify the busiest days for the helpdesk. Staffing plans should match the days when ticket demand is highest.

In [ ]:
tickets_by_hour = df.groupby("Heure_Ouverture").size().sort_index()

plt.figure(figsize=(10, 5))
plt.bar(tickets_by_hour.index, tickets_by_hour.values, color="#E76F51")
plt.title("Tickets by Opening Hour")
plt.xlabel("Opening Hour")
plt.ylabel("Number of Tickets")
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

**Business interpretation:** Hourly ticket patterns show peak request periods during the day. These peaks can guide shift planning and helpdesk availability.

## Final Notes

This EDA highlights workload distribution, SLA performance, technician activity, satisfaction patterns, and time-based demand. The insights from this notebook can support dashboard creation and business recommendations in the next project phases.